In [ ]:
%%sh
set -euo pipefail

OUTDIR_BASE="/data_disk/liling/fig6" 
PAIRS="/data_disk/data/liling/scMicroC_data/txt"
META="/data_disk/data/liling/scMicroC_data/metadata.tsv"

CHR1=1; POS1=2; CHR2=3; POS2=4

RUNS=(
  "0.0  0.20 0_20per"
  "0.20 0.40 20_40per"
  "0.40 0.50 40_50per"
  "0.50 0.60 50_60per"
  "0.60 0.70 60_70per"
  "0.70 0.80 70_80per"
)

run_one () {
  flank="$1"; q1="$2"; q2="$3"; tag="$4"

  OUTDIR="${OUTDIR_BASE}/${flank}_EBF1_submat_5kb_wid6"
  STRIPE="${OUTDIR}/processed_results.tsv"

  mkdir -p "${OUTDIR}"

  echo "=== flanking=${flank} q1=${q1} q2=${q2} tag=${tag} ==="

  scstripe score-cells \
    --q1 "${q1}" --q2 "${q2}" \
    --flanking "${flank}" \
    --celltype scMicroC \
    --pairs-dir "${PAIRS}" \
    --meta-file "${META}" \
    --output-dir "${OUTDIR}" \
    --stripe-file "${STRIPE}" \
    --flank-dis-decay \
    --chr1-col ${CHR1} --pos1-col ${POS1} --chr2-col ${CHR2} --pos2-col ${POS2}

  src="${OUTDIR}/stripe_score_scMicroC.tsv"
  dst="${OUTDIR}/stripe_score_scMicroC_${tag}.tsv"
  if [ ! -s "${src}" ]; then
    echo "ERROR: expected output not found: ${src}"
    ls -lh "${OUTDIR}" | tail -n 50
    exit 1
  fi
  mv "${src}" "${dst}"
}

for flank in both; do
  for item in "${RUNS[@]}"; do
    run_one ${flank} ${item}
  done
done

# hierarchical

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

out = "/data_disk/liling/fig6/both_EBF1_submat_5kb_wid6"
FILES = [
    f"{out}/stripe_score_scMicroC_0_20per.tsv",
    f"{out}/stripe_score_scMicroC_20_40per.tsv",
    f"{out}/stripe_score_scMicroC_40_50per.tsv",
    f"{out}/stripe_score_scMicroC_50_60per.tsv",
    f"{out}/stripe_score_scMicroC_60_70per.tsv",
    f"{out}/stripe_score_scMicroC_70_80per.tsv",
]
N_META_COLS = 9
DROP_LAST_COL = True
COND_REGEX = re.compile(r"_(\d+_\d+per)\.tsv$")

N_CLUSTERS = 13
HCLUST_METHOD = "ward"  
HCLUST_METRIC = "euclidean" 

OUT_DIR = f"{out}/hierarchical_0_20_40_50_60_70_80"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_MATRIX_TSV = os.path.join(OUT_DIR, "merged_cell_by_condition.tsv")
OUT_LABELS_TSV = os.path.join(OUT_DIR, "hclust_labels.tsv")  
OUT_SORTED_TSV = os.path.join(OUT_DIR, "merged_sorted_by_cluster.tsv")
OUT_HEATMAP_PNG = os.path.join(OUT_DIR, "heatmap_by_cluster_hclust.png")
OUT_HEATMAP_PDF = os.path.join(OUT_DIR, "heatmap_by_cluster_hclust.pdf")


series_list = []
for fp in FILES:
    fname = os.path.basename(fp)
    m = COND_REGEX.search(fname) or re.search(r"(\d+_\d+per)\.tsv$", fname)
    cond = m.group(1)  

    df = pd.read_csv(fp, sep="\t", header=0)
    need_min_cols = N_META_COLS + (1 if DROP_LAST_COL else 0) + 1
    df_cells = df.iloc[:, N_META_COLS:-1] if DROP_LAST_COL else df.iloc[:, N_META_COLS:]

    for c in df_cells.columns:
        if not np.issubdtype(df_cells[c].dtype, np.number):
            df_cells[c] = pd.to_numeric(df_cells[c], errors="coerce")

    s = df_cells.mean(axis=0, skipna=True)
    s.name = cond
    series_list.append(s)

merged = pd.concat(series_list, axis=1)

def _cond_key(c):
    m = re.match(r"(\d+)_(\d+)per", c)
    return (int(m.group(1)), int(m.group(2))) if m else (10**9, 10**9)

merged = merged.loc[sorted(merged.index), sorted(merged.columns, key=_cond_key)]
merged.to_csv(OUT_MATRIX_TSV, sep="\t")

X = merged.replace([np.inf, -np.inf], np.nan)
X = X.apply(lambda col: col.fillna(col.mean()), axis=0)  

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values) 

Z = linkage(X_scaled, method=HCLUST_METHOD, metric=HCLUST_METRIC)
leaf_order = dendrogram(Z, no_plot=True)["leaves"]

raw_labels = fcluster(Z, t=N_CLUSTERS, criterion="maxclust")

ordered_cells_by_tree = X.index.to_numpy()[leaf_order]

label_series = pd.Series(raw_labels, index=X.index, name="cluster")

df_plot = merged.copy()          
df_plot["cluster"] = label_series
df_plot["row_mean"] = df_plot.drop(columns=["cluster"]).mean(axis=1)

unique_labels = np.sort(df_plot["cluster"].unique())
label_remap = {lab: i for i, lab in enumerate(unique_labels)}
df_plot["cluster"] = df_plot["cluster"].map(label_remap)

df_plot = df_plot.sort_values(by=["cluster", "row_mean"], ascending=[True, False])

ordered_cells = df_plot.index.tolist()
ordered_conditions = list(merged.columns)

df_sorted = merged.loc[ordered_cells, ordered_conditions]  
df_sorted.to_csv(OUT_SORTED_TSV, sep="\t")

label_out = pd.DataFrame({"cell": merged.index, "cluster": label_series.map(label_remap)}).set_index("cell")
label_out.to_csv(OUT_LABELS_TSV, sep="\t")

cluster_order = df_plot["cluster"].to_numpy()
bounds = []
spans = []  # (cluster_id, start_row, end_row, size)
if len(cluster_order) > 0:
    start = 0
    current = cluster_order[0]
    for i in range(1, len(cluster_order)):
        if cluster_order[i] != current:
            end = i - 1
            spans.append((int(current), start, end, end - start + 1))
            bounds.append(i)
            start = i
            current = cluster_order[i]
    end = len(cluster_order) - 1
    spans.append((int(current), start, end, end - start + 1))

disp = df_sorted.values.astype(float)  
n_rows, n_cols = disp.shape
fig_h = min(max(n_rows / 60.0, 4.5), 18.0)
fig_w = min(max(n_cols / 4.0, 6.0), 18.0)

data_min = float(np.nanmin(disp)) if disp.size else 0.0
data_max = float(np.nanmax(disp)) if disp.size else 1.0
if not (data_min < 1.0 < data_max):
    span = max(abs(1.0 - data_min), abs(data_max - 1.0))
    vmin, vmax = 1.0 - span, 1.0 + span
else:
    vmin, vmax = data_min, data_max
norm = TwoSlopeNorm(vmin=vmin, vcenter=1.0, vmax=vmax)
cmap = mpl.cm.get_cmap("bwr")

fig = plt.figure(figsize=(fig_w, fig_h), dpi=180)
gs = fig.add_gridspec(nrows=1, ncols=2, width_ratios=[0.4, 12], wspace=0.05)

ax_bar = fig.add_subplot(gs[0, 0])
cluster_img = cluster_order.reshape(-1, 1)
cmap_clusters = mpl.cm.get_cmap("tab10", len(unique_labels))
ax_bar.imshow(cluster_img, aspect="auto", interpolation="nearest", cmap=cmap_clusters)
ax_bar.set_xticks([])
ax_bar.set_yticks([])
ax_bar.set_title("Cluster", fontsize=9, pad=6)

ax = fig.add_subplot(gs[0, 1])
im = ax.imshow(disp, aspect="auto", interpolation="nearest", cmap=cmap, norm=norm)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("value (center = 1)")

ax.set_yticks([])  
ax.set_xticks(np.arange(n_cols))
ax.set_xticklabels(ordered_conditions, rotation=60, ha="right", fontsize=9)

ax.set_xlabel("Condition")
ax.set_ylabel("Cells (sorted by hierarchical clusters)")
ax.set_title(f"Stripe-score heatmap by Hierarchical Clustering (K={N_CLUSTERS})")


for b in bounds:
    ax.axhline(b - 0.5, linewidth=1.6, linestyle="--", color="k", alpha=0.8)
    ax_bar.axhline(b - 0.5, linewidth=1.6, linestyle="--", color="k", alpha=0.8)


for cid, s, e, size in spans:
    y_mid = (s + e) / 2.0
    ax.text(n_cols + 0.15, y_mid, f"C{cid} (n={size})", va="center", fontsize=9)
ax.set_xlim(-0.5, n_cols + 1.5)

plt.tight_layout()
fig.savefig(OUT_HEATMAP_PNG)
fig.savefig(OUT_HEATMAP_PDF)
plt.close(fig)